# 🏭 Tối Ưu Hóa Hệ Thống Quản Lý Kho Hàng
## Đồ án Big Data - Sử dụng PySpark + K-Means Clustering

**Đề tài:** Sử dụng dữ liệu lớn để tối ưu hóa quản lý kho và phân phối sản phẩm

**Công nghệ:** Python, PySpark, Hadoop (HDFS), Docker, Power BI

**Dataset:** Brazilian E-Commerce (Olist) - Kaggle

---
### Các bước thực hiện:
1. Cài đặt & Thu thập dữ liệu
2. Làm sạch dữ liệu (Data Cleaning)
3. Join bảng & Tính toán doanh thu
4. Machine Learning - K-Means Clustering
5. Trực quan hóa kết quả
6. Tối ưu hóa Spark
7. Export cho Power BI

## 1. Cài đặt môi trường & Tải dữ liệu

In [ ]:
# Cài đặt PySpark trên Google Colab
!pip install pyspark==3.5.0 kaggle -q

In [ ]:
# Tải dataset từ Kaggle
# Cách 1: Upload kaggle.json rồi chạy
# from google.colab import files
# files.upload()  # Upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d olistbr/brazilian-ecommerce -p data/raw --unzip

# Cách 2: Tải thủ công từ Kaggle và upload lên Colab
# Hoặc mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Nếu đã tải dataset vào Drive:
# DATA_PATH = "/content/drive/MyDrive/BigData/data/raw"

# Hoặc dùng Kaggle API:
import os
os.makedirs("data/raw", exist_ok=True)
os.makedirs("output/powerbi", exist_ok=True)

!kaggle datasets download -d olistbr/brazilian-ecommerce -p data/raw --unzip

DATA_PATH = "data/raw"
print("Files downloaded:")
!ls -la data/raw/

In [ ]:
# Khởi tạo SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline

import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Khởi tạo Spark
spark = (SparkSession.builder
    .appName("WarehouseOptimization")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: http://localhost:4040")

## 2. Load & Phân tích cấu trúc dữ liệu

In [ ]:
# Load tất cả các bảng
print("=" * 60)
print("  LOADING DATA")
print("=" * 60)

tables = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

raw = {}
for name, filename in tables.items():
    filepath = f"{DATA_PATH}/{filename}"
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    raw[name] = df
    print(f"  {name}: {df.count():,} rows, {len(df.columns)} columns")

print(f"\nTổng: {len(raw)} bảng dữ liệu")

In [ ]:
# Xem cấu trúc từng bảng
for name, df in raw.items():
    print(f"\n{'='*40} {name.upper()} {'='*40}")
    df.printSchema()
    df.show(3, truncate=False)

In [ ]:
# Phân tích Missing Values
print("=" * 60)
print("  MISSING VALUES ANALYSIS")
print("=" * 60)

for name, df in raw.items():
    total_rows = df.count()
    has_missing = False
    for column in df.columns:
        null_count = df.filter(col(column).isNull()).count()
        if null_count > 0:
            if not has_missing:
                print(f"\n--- {name} ({total_rows:,} rows) ---")
                has_missing = True
            pct = (null_count / total_rows) * 100
            print(f"  {column}: {null_count:,} ({pct:.1f}%)")

## 3. Làm sạch dữ liệu (Data Cleaning)

In [ ]:
# ====== 3.1 Clean Orders ======
print("Cleaning: orders")
df_orders = raw["orders"]

# Chuyển đổi timestamp
timestamp_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for c in timestamp_cols:
    df_orders = df_orders.withColumn(c, to_timestamp(col(c)))

# Loại bỏ duplicate
before = df_orders.count()
df_orders = df_orders.dropDuplicates(["order_id"])
print(f"  Removed {before - df_orders.count()} duplicate orders")

# Tính thời gian giao hàng
df_orders = (df_orders
    .withColumn("delivery_days",
        datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
    .withColumn("delivery_delay_days",
        datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")))
)

print(f"  Orders: {df_orders.count():,}")
print(f"  Delivered: {df_orders.filter(col('order_status')=='delivered').count():,}")
df_orders.show(3)

In [ ]:
# ====== 3.2 Clean Products ======
print("Cleaning: products")
df_products = raw["products"]
df_translation = raw["category_translation"]

# Join với bảng dịch category
df_products = df_products.join(df_translation, "product_category_name", "left")

# Fill missing values
df_products = (df_products
    .withColumn("product_category_name_english",
        when(col("product_category_name_english").isNull(), lit("unknown"))
        .otherwise(col("product_category_name_english")))
    .na.fill(0, ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"])
)

# Tính thể tích
df_products = df_products.withColumn(
    "product_volume_cm3",
    col("product_length_cm") * col("product_height_cm") * col("product_width_cm")
)

df_products = df_products.dropDuplicates(["product_id"])
print(f"  Products: {df_products.count():,}")
df_products.show(3)

In [ ]:
# ====== 3.3 Clean Order Items ======
print("Cleaning: order_items")
df_items = raw["order_items"]

# Tính tổng giá trị
df_items = df_items.withColumn(
    "total_item_value",
    round(col("price") + col("freight_value"), 2)
)
df_items = df_items.withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date")))

# Loại bỏ giá trị bất thường
before = df_items.count()
df_items = df_items.filter(col("price") > 0)
print(f"  Removed {before - df_items.count()} items with invalid price")
print(f"  Order items: {df_items.count():,}")

# ====== 3.4 Clean Payments ======
print("\nCleaning: payments")
df_payments = raw["payments"].filter(col("payment_value") > 0)
print(f"  Payments: {df_payments.count():,}")

# Các bảng khác
df_sellers = raw["sellers"]
df_customers = raw["customers"]
df_geolocation = raw["geolocation"].dropDuplicates(["geolocation_zip_code_prefix"])

print("\n✅ Data Cleaning Complete!")

## 4. Join các bảng & Tính toán doanh thu

In [ ]:
# ====== JOIN TẤT CẢ CÁC BẢNG ======
print("=" * 60)
print("  JOINING TABLES")
print("=" * 60)

# 1. Aggregate payments per order
payments_agg = (df_payments
    .groupBy("order_id")
    .agg(
        sum("payment_value").alias("total_payment"),
        countDistinct("payment_type").alias("payment_methods_count"),
        first("payment_type").alias("primary_payment_type"),
        sum("payment_installments").alias("total_installments")
    )
)

# 2. order_items + products
items_products = df_items.join(df_products, "product_id", "left")
print(f"  order_items + products: {items_products.count():,} rows")

# 3. + sellers
items_full = items_products.join(df_sellers, "seller_id", "left")

# 4. + orders
orders_items = df_orders.join(items_full, "order_id", "inner")
print(f"  + orders: {orders_items.count():,} rows")

# 5. + customers
full_df = orders_items.join(df_customers, "customer_id", "left")

# 6. + payments
full_df = full_df.join(payments_agg, "order_id", "left")

# 7. Thêm cột thời gian
full_df = (full_df
    .withColumn("order_year", year("order_purchase_timestamp"))
    .withColumn("order_month", month("order_purchase_timestamp"))
    .withColumn("order_dayofweek", dayofweek("order_purchase_timestamp"))
    .withColumn("order_yearmonth", date_format("order_purchase_timestamp", "yyyy-MM"))
)

print(f"\n  ✅ Final table: {full_df.count():,} rows, {len(full_df.columns)} columns")
full_df.printSchema()

In [ ]:
# Cache bảng chính để tăng tốc
full_df.cache()
full_df.count()  # trigger cache
print("✅ Data cached!")

In [ ]:
# ====== TÍNH TOÁN DOANH THU ======
print("=" * 60)
print("  REVENUE METRICS")
print("=" * 60)

# 1. Doanh thu theo tháng
monthly_revenue = (full_df
    .groupBy("order_yearmonth")
    .agg(
        sum("total_item_value").alias("revenue"),
        count("order_id").alias("num_orders"),
        countDistinct("customer_unique_id").alias("unique_customers"),
        avg("total_item_value").alias("avg_order_value"),
        sum("freight_value").alias("total_freight")
    )
    .orderBy("order_yearmonth")
)

print("\n📊 Doanh thu theo tháng:")
monthly_revenue.show(24, truncate=False)

In [ ]:
# 2. Doanh thu theo danh mục sản phẩm (Top 15)
category_revenue = (full_df
    .groupBy("product_category_name_english")
    .agg(
        sum("total_item_value").alias("revenue"),
        count("*").alias("num_items_sold"),
        countDistinct("product_id").alias("unique_products"),
        avg("price").alias("avg_price"),
        avg("freight_value").alias("avg_freight"),
        avg("product_weight_g").alias("avg_weight_g")
    )
    .orderBy(col("revenue").desc())
)

print("📊 Top 15 Categories by Revenue:")
category_revenue.show(15, truncate=False)

In [ ]:
# 3. Doanh thu theo khu vực seller
seller_state_revenue = (full_df
    .groupBy("seller_state")
    .agg(
        sum("total_item_value").alias("revenue"),
        count("*").alias("num_items_sold"),
        countDistinct("seller_id").alias("num_sellers"),
        avg("delivery_days").alias("avg_delivery_days")
    )
    .orderBy(col("revenue").desc())
)

print("📊 Revenue by Seller State:")
seller_state_revenue.show(15)

# 4. Doanh thu theo khu vực khách hàng
customer_state_revenue = (full_df
    .groupBy("customer_state")
    .agg(
        sum("total_item_value").alias("revenue"),
        countDistinct("customer_unique_id").alias("num_customers"),
        avg("total_item_value").alias("avg_order_value")
    )
    .orderBy(col("revenue").desc())
)

print("📊 Revenue by Customer State:")
customer_state_revenue.show(15)

In [ ]:
# 5. Phân tích hiệu suất giao hàng
delivery_performance = (full_df
    .filter(col("order_status") == "delivered")
    .groupBy("seller_state")
    .agg(
        avg("delivery_days").alias("avg_delivery_days"),
        avg("delivery_delay_days").alias("avg_delay_days"),
        sum(when(col("delivery_delay_days") > 0, 1).otherwise(0)).alias("late_deliveries"),
        count("*").alias("total_deliveries")
    )
    .withColumn("late_delivery_rate",
        round(col("late_deliveries") / col("total_deliveries") * 100, 2))
    .orderBy(col("total_deliveries").desc())
)

print("📊 Delivery Performance by State:")
delivery_performance.show(15)

# 6. Seller performance
seller_performance = (full_df
    .groupBy("seller_id", "seller_city", "seller_state")
    .agg(
        sum("total_item_value").alias("revenue"),
        count("*").alias("items_sold"),
        countDistinct("product_id").alias("unique_products"),
        countDistinct("order_id").alias("num_orders"),
        avg("delivery_days").alias("avg_delivery_days")
    )
    .orderBy(col("revenue").desc())
)

print("📊 Top 10 Sellers:")
seller_performance.show(10)

In [ ]:
# 7. Phân tích ABC cho sản phẩm
product_revenue = (full_df
    .groupBy("product_id", "product_category_name_english")
    .agg(
        sum("total_item_value").alias("revenue"),
        count("*").alias("quantity_sold"),
        avg("freight_value").alias("avg_freight")
    )
)

total_rev = product_revenue.agg(sum("revenue")).collect()[0][0]

window = Window.orderBy(col("revenue").desc())
product_abc = (product_revenue
    .withColumn("rank", dense_rank().over(window))
    .withColumn("pct_rank", percent_rank().over(window))
    .withColumn("revenue_pct", round(col("revenue") / lit(total_rev) * 100, 2))
    .withColumn("abc_class",
        when(col("pct_rank") <= 0.2, "A")
        .when(col("pct_rank") <= 0.5, "B")
        .otherwise("C")
    )
)

print("📊 ABC Analysis Summary:")
abc_summary = (product_abc
    .groupBy("abc_class")
    .agg(
        count("*").alias("num_products"),
        sum("revenue").alias("total_revenue"),
        sum("quantity_sold").alias("total_quantity")
    )
    .orderBy("abc_class")
)
abc_summary.show()

## 5. Trực quan hóa dữ liệu

In [ ]:
# ====== TRỰC QUAN HÓA ======
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

# 1. Biểu đồ doanh thu theo tháng
pdf_monthly = monthly_revenue.toPandas()

fig, ax1 = plt.subplots(figsize=(16, 6))
ax1.bar(pdf_monthly['order_yearmonth'], pdf_monthly['revenue'], 
        color='#4ECDC4', alpha=0.7, label='Revenue')
ax1.set_xlabel('Tháng')
ax1.set_ylabel('Doanh thu (BRL)', color='#4ECDC4')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(pdf_monthly['order_yearmonth'], pdf_monthly['num_orders'], 
         'ro-', linewidth=2, label='Orders')
ax2.set_ylabel('Số đơn hàng', color='red')

plt.title('Doanh thu & Số đơn hàng theo tháng', fontsize=16, fontweight='bold')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.95))
plt.tight_layout()
plt.savefig('output/monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2. Top 15 danh mục sản phẩm
pdf_cat = category_revenue.limit(15).toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Bar chart doanh thu
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(pdf_cat)))
ax1.barh(pdf_cat['product_category_name_english'][::-1], 
         pdf_cat['revenue'][::-1], color=colors)
ax1.set_xlabel('Doanh thu (BRL)')
ax1.set_title('Top 15 Danh mục theo doanh thu')
ax1.grid(axis='x', alpha=0.3)

# Bar chart số lượng bán
ax2.barh(pdf_cat['product_category_name_english'][::-1], 
         pdf_cat['num_items_sold'][::-1], color=colors)
ax2.set_xlabel('Số lượng bán')
ax2.set_title('Top 15 Danh mục theo số lượng')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('output/top_categories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3. Doanh thu theo khu vực
pdf_state = seller_state_revenue.toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 states
top10 = pdf_state.head(10)
ax1.bar(top10['seller_state'], top10['revenue'], color='#FF6B6B', alpha=0.8)
ax1.set_xlabel('State')
ax1.set_ylabel('Doanh thu (BRL)')
ax1.set_title('Doanh thu theo bang (Seller)')
ax1.grid(axis='y', alpha=0.3)

# Pie chart
top5 = pdf_state.head(5)
others = pd.DataFrame({'seller_state': ['Others'], 'revenue': [pdf_state['revenue'][5:].sum()]})
pie_data = pd.concat([top5[['seller_state', 'revenue']], others])
ax2.pie(pie_data['revenue'], labels=pie_data['seller_state'], 
        autopct='%1.1f%%', colors=['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4','#FFEAA7','#DFE6E9'])
ax2.set_title('Tỉ lệ doanh thu Top 5 States')

plt.tight_layout()
plt.savefig('output/revenue_by_state.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4. Phân tích giao hàng
pdf_delivery = delivery_performance.toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Avg delivery days
top_states = pdf_delivery.head(10)
ax1.bar(top_states['seller_state'], top_states['avg_delivery_days'], 
        color='#45B7D1', alpha=0.8)
ax1.set_xlabel('State')
ax1.set_ylabel('Ngày giao TB')
ax1.set_title('Thời gian giao hàng trung bình')
ax1.grid(axis='y', alpha=0.3)

# Late delivery rate
ax2.bar(top_states['seller_state'], top_states['late_delivery_rate'], 
        color='#FF6B6B', alpha=0.8)
ax2.set_xlabel('State')
ax2.set_ylabel('Tỉ lệ trễ (%)')
ax2.set_title('Tỉ lệ giao hàng trễ')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('output/delivery_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5. ABC Analysis visualization
pdf_abc = abc_summary.toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors_abc = ['#FF6B6B', '#FFEAA7', '#4ECDC4']

ax1.bar(pdf_abc['abc_class'], pdf_abc['num_products'], color=colors_abc)
ax1.set_xlabel('Lớp ABC')
ax1.set_ylabel('Số sản phẩm')
ax1.set_title('Phân tích ABC - Số lượng sản phẩm')
for i, v in enumerate(pdf_abc['num_products']):
    ax1.text(i, v + 50, str(v), ha='center', fontweight='bold')

ax2.bar(pdf_abc['abc_class'], pdf_abc['total_revenue'], color=colors_abc)
ax2.set_xlabel('Lớp ABC')
ax2.set_ylabel('Tổng doanh thu (BRL)')
ax2.set_title('Phân tích ABC - Doanh thu')

plt.tight_layout()
plt.savefig('output/abc_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Machine Learning - K-Means Clustering

In [ ]:
# ====== CHUẨN BỊ FEATURES ======
print("=" * 60)
print("  K-MEANS CLUSTERING")
print("=" * 60)

# Aggregate features theo product_id
product_features = (full_df
    .groupBy("product_id")
    .agg(
        sum("total_item_value").alias("total_revenue"),
        count("*").alias("times_sold"),
        avg("price").alias("avg_price"),
        avg("freight_value").alias("avg_freight"),
        first("product_weight_g").alias("weight_g"),
        first("product_volume_cm3").alias("volume_cm3"),
        first("product_category_name_english").alias("category"),
        avg("delivery_days").alias("avg_delivery_days")
    )
)

# Fill null, filter
product_features = (product_features
    .na.fill(0)
    .filter(col("total_revenue") > 0)
    .filter(col("times_sold") > 0)
)

# Log transform cho features skewed
product_features = (product_features
    .withColumn("log_revenue", log(col("total_revenue") + 1))
    .withColumn("log_times_sold", log(col("times_sold") + 1))
    .withColumn("log_weight", log(col("weight_g") + 1))
    .withColumn("log_volume", log(col("volume_cm3") + 1))
)

print(f"Products with features: {product_features.count():,}")
product_features.describe().show()

In [ ]:
# ====== TÌM K TỐI ƯU ======
feature_cols = ["log_revenue", "log_times_sold", "avg_price", 
                "avg_freight", "log_weight", "log_volume"]

print("Finding optimal K...")
k_range = range(2, 11)
scores = []
costs = []

for k in k_range:
    # Pipeline: VectorAssembler -> StandardScaler -> KMeans
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
    scaler = StandardScaler(inputCol="features_raw", outputCol="features", 
                           withStd=True, withMean=True)
    kmeans = KMeans(featuresCol="features", predictionCol="cluster", 
                    k=k, seed=42, maxIter=100)
    
    pipeline = Pipeline(stages=[assembler, scaler, kmeans])
    model = pipeline.fit(product_features)
    predictions = model.transform(product_features)
    
    # Silhouette Score
    evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="cluster")
    score = evaluator.evaluate(predictions)
    scores.append(score)
    
    # WSSSE
    cost = model.stages[-1].summary.trainingCost
    costs.append(cost)
    
    print(f"  K={k}: Silhouette={score:.4f}, WSSSE={cost:.2f}")

best_k = list(k_range)[np.argmax(scores)]
print(f"\n✅ Best K = {best_k} (Silhouette = {max(scores):.4f})")

In [ ]:
# Vẽ biểu đồ Elbow & Silhouette
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(list(k_range), costs, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Số cụm (K)', fontsize=12)
ax1.set_ylabel('WSSSE', fontsize=12)
ax1.set_title('Elbow Method', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(list(k_range), scores, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Số cụm (K)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title('Silhouette Score theo K', fontsize=14, fontweight='bold')
ax2.axvline(x=best_k, color='green', linestyle='--', linewidth=2, 
            label=f'Best K={best_k}')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('output/elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====== TRAIN MODEL VỚI K TỐI ƯU ======
print(f"\nTraining K-Means with K={best_k}...")

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features",
                       withStd=True, withMean=True)
kmeans = KMeans(featuresCol="features", predictionCol="cluster",
                k=best_k, seed=42, maxIter=100)

pipeline = Pipeline(stages=[assembler, scaler, kmeans])
model = pipeline.fit(product_features)
predictions = model.transform(product_features)

# Đánh giá
evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="cluster")
silhouette = evaluator.evaluate(predictions)
print(f"Silhouette Score: {silhouette:.4f}")

# Thống kê từng cụm
print("\n📊 Cluster Summary:")
cluster_summary = (predictions
    .groupBy("cluster")
    .agg(
        count("*").alias("num_products"),
        round(avg("total_revenue"), 2).alias("avg_revenue"),
        round(avg("times_sold"), 1).alias("avg_times_sold"),
        round(avg("avg_price"), 2).alias("avg_price"),
        round(avg("weight_g"), 0).alias("avg_weight"),
        round(avg("avg_freight"), 2).alias("avg_freight")
    )
    .orderBy("cluster")
)
cluster_summary.show(truncate=False)

In [ ]:
# ====== GÁN NHÃN CỤM ======
cluster_stats = (predictions
    .groupBy("cluster")
    .agg(avg("total_revenue").alias("avg_rev"), avg("times_sold").alias("avg_sold"))
    .collect()
)

sorted_clusters = sorted(cluster_stats, key=lambda x: x["avg_rev"], reverse=True)

labels = [
    "Premium - Doanh thu cao",
    "Standard - Bán chạy",
    "Economy - Trung bình",
    "Low-value - Ít bán"
]

# Tạo mapping
conditions = None
for i, cluster_row in enumerate(sorted_clusters):
    label = labels[i] if i < len(labels) else f"Cluster {cluster_row['cluster']}"
    if conditions is None:
        conditions = when(col("cluster") == cluster_row["cluster"], lit(label))
    else:
        conditions = conditions.when(col("cluster") == cluster_row["cluster"], lit(label))

predictions = predictions.withColumn("cluster_label", conditions.otherwise(lit("Other")))

print("📊 Cluster Labels:")
predictions.groupBy("cluster", "cluster_label").count().orderBy("cluster").show(truncate=False)

In [ ]:
# ====== TRỰC QUAN HÓA CLUSTERING ======
pdf = predictions.toPandas()

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DFE6E9']

for cluster_id in sorted(pdf['cluster'].unique()):
    mask = pdf['cluster'] == cluster_id
    label = pdf[mask]['cluster_label'].iloc[0]
    c = colors[cluster_id % len(colors)]
    
    axes[0,0].scatter(pdf[mask]['log_revenue'], pdf[mask]['log_times_sold'],
                      c=c, label=label, alpha=0.6, s=30)
    axes[0,1].scatter(pdf[mask]['avg_price'], pdf[mask]['avg_freight'],
                      c=c, label=label, alpha=0.6, s=30)
    axes[1,0].scatter(pdf[mask]['log_weight'], pdf[mask]['log_revenue'],
                      c=c, label=label, alpha=0.6, s=30)

axes[0,0].set_xlabel('Log Doanh thu')
axes[0,0].set_ylabel('Log Số lần bán')
axes[0,0].set_title('Phân cụm: Doanh thu vs Số lần bán', fontweight='bold')
axes[0,0].legend(fontsize=8)
axes[0,0].grid(True, alpha=0.3)

axes[0,1].set_xlabel('Giá TB (BRL)')
axes[0,1].set_ylabel('Phí vận chuyển TB (BRL)')
axes[0,1].set_title('Phân cụm: Giá vs Phí vận chuyển', fontweight='bold')
axes[0,1].legend(fontsize=8)
axes[0,1].grid(True, alpha=0.3)

axes[1,0].set_xlabel('Log Trọng lượng')
axes[1,0].set_ylabel('Log Doanh thu')
axes[1,0].set_title('Phân cụm: Trọng lượng vs Doanh thu', fontweight='bold')
axes[1,0].legend(fontsize=8)
axes[1,0].grid(True, alpha=0.3)

# Pie chart
cluster_counts = pdf.groupby('cluster_label')['product_id'].count()
axes[1,1].pie(cluster_counts.values, labels=cluster_counts.index,
              autopct='%1.1f%%', colors=colors[:len(cluster_counts)], startangle=90)
axes[1,1].set_title('Phân bổ sản phẩm theo cụm', fontweight='bold')

plt.suptitle('Kết quả K-Means Clustering', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/clustering_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Box plots cho từng feature theo cụm
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
features_plot = ['total_revenue', 'times_sold', 'avg_price', 'avg_freight', 'weight_g', 'avg_delivery_days']
titles_plot = ['Doanh thu', 'Số lần bán', 'Giá TB', 'Phí vận chuyển', 'Trọng lượng (g)', 'Ngày giao TB']

for idx, (feat, title) in enumerate(zip(features_plot, titles_plot)):
    ax = axes[idx // 3][idx % 3]
    if feat in pdf.columns:
        data = [pdf[pdf['cluster'] == c][feat].dropna().values 
                for c in sorted(pdf['cluster'].unique())]
        bp = ax.boxplot(data, patch_artist=True)
        for i, patch in enumerate(bp['boxes']):
            patch.set_facecolor(colors[i % len(colors)])
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Cluster')
        ax.grid(True, alpha=0.3)

plt.suptitle('Phân bổ Features theo Cluster', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('output/cluster_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top categories theo cluster
print("📊 Top 5 Categories per Cluster:")
for cluster_id in sorted(pdf['cluster'].unique()):
    label = pdf[pdf['cluster'] == cluster_id]['cluster_label'].iloc[0]
    print(f"\n--- Cluster {cluster_id}: {label} ---")
    top_cats = (predictions
        .filter(col("cluster") == cluster_id)
        .groupBy("category")
        .agg(count("*").alias("count"), sum("total_revenue").alias("revenue"))
        .orderBy(col("revenue").desc())
        .limit(5)
    )
    top_cats.show(truncate=False)

## 7. Tối ưu hóa Spark

In [ ]:
# ====== DEMO TỐI ƯU HÓA SPARK ======
import time

print("=" * 60)
print("  SPARK OPTIMIZATION TECHNIQUES")
print("=" * 60)

# Unpersist trước khi test
full_df.unpersist()

# --- 1. CACHE / PERSIST ---
print("\n--- 1. Cache / Persist ---")
start = time.time()
full_df.groupBy("product_category_name_english").count().collect()
full_df.groupBy("seller_state").count().collect()
no_cache_time = time.time() - start
print(f"  Without cache: {no_cache_time:.2f}s")

full_df.cache()
full_df.count()

start = time.time()
full_df.groupBy("product_category_name_english").count().collect()
full_df.groupBy("seller_state").count().collect()
cache_time = time.time() - start
print(f"  With cache:    {cache_time:.2f}s")
print(f"  Speedup:       {no_cache_time / max(cache_time, 0.001):.1f}x")

In [ ]:
# --- 2. REPARTITION ---
print("\n--- 2. Repartition ---")
print(f"  Current partitions: {full_df.rdd.getNumPartitions()}")

start = time.time()
df_repartitioned = full_df.repartition(8, "seller_state")
df_repartitioned.groupBy("seller_state").count().collect()
repart_time = time.time() - start
print(f"  After repartition: {repart_time:.2f}s")
print(f"  New partitions: {df_repartitioned.rdd.getNumPartitions()}")

In [ ]:
# --- 3. BROADCAST JOIN ---
print("\n--- 3. Broadcast Join ---")
small_df = full_df.select("seller_state").distinct()
print(f"  Small table rows: {small_df.count()}")

start = time.time()
full_df.join(small_df, "seller_state", "inner").count()
normal_join = time.time() - start
print(f"  Normal join:    {normal_join:.2f}s")

start = time.time()
full_df.join(broadcast(small_df), "seller_state", "inner").count()
broadcast_time = time.time() - start
print(f"  Broadcast join: {broadcast_time:.2f}s")
print(f"  Speedup:        {normal_join / max(broadcast_time, 0.001):.1f}x")

In [ ]:
# --- 4. EXPLAIN PLAN ---
print("\n--- 4. Query Execution Plan ---")
query = (full_df
    .filter(col("order_status") == "delivered")
    .groupBy("product_category_name_english")
    .agg({"total_item_value": "sum", "order_id": "count"})
    .orderBy(col("sum(total_item_value)").desc())
    .limit(10)
)
query.explain(mode="simple")

In [ ]:
# Biểu đồ so sánh performance
fig, ax = plt.subplots(figsize=(10, 6))

techniques = ['No Cache', 'With Cache', 'Normal Join', 'Broadcast Join']
times = [no_cache_time, cache_time, normal_join, broadcast_time]
bar_colors = ['#FF6B6B', '#4ECDC4', '#FF6B6B', '#4ECDC4']

bars = ax.bar(techniques, times, color=bar_colors, alpha=0.8)
ax.set_ylabel('Thời gian (giây)', fontsize=12)
ax.set_title('So sánh Performance - Spark Optimization', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
            f'{t:.2f}s', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('output/spark_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Export dữ liệu cho Power BI

In [ ]:
# ====== EXPORT CHO POWER BI ======
import os
os.makedirs("output/powerbi", exist_ok=True)

print("=" * 60)
print("  EXPORTING DATA FOR POWER BI")
print("=" * 60)

exports = {
    "monthly_revenue": monthly_revenue,
    "category_revenue": category_revenue,
    "seller_state_revenue": seller_state_revenue,
    "customer_state_revenue": customer_state_revenue,
    "delivery_performance": delivery_performance,
    "seller_performance": seller_performance.limit(500),
}

for name, df in exports.items():
    filepath = f"output/powerbi/{name}.csv"
    df.toPandas().to_csv(filepath, index=False)
    print(f"  Exported: {name}.csv")

# Clustering results
cluster_export = predictions.select(
    "product_id", "category", "total_revenue", "times_sold",
    "avg_price", "avg_freight", "weight_g", "volume_cm3",
    "cluster", "cluster_label"
)
cluster_export.toPandas().to_csv("output/powerbi/product_clusters.csv", index=False)
print("  Exported: product_clusters.csv")

# Fact table
fact_cols = [
    "order_id", "customer_unique_id", "customer_state", "customer_city",
    "order_yearmonth", "order_year", "order_month", "order_dayofweek",
    "order_status", "product_id", "product_category_name_english",
    "seller_id", "seller_state", "seller_city",
    "price", "freight_value", "total_item_value",
    "delivery_days", "delivery_delay_days",
    "primary_payment_type", "total_payment"
]
existing_cols = [c for c in fact_cols if c in full_df.columns]
full_df.select(existing_cols).toPandas().to_csv("output/powerbi/fact_orders.csv", index=False)
print("  Exported: fact_orders.csv")

print("\n✅ All files exported to output/powerbi/")
print("   Import these CSVs into Power BI Desktop")
!ls -la output/powerbi/

In [ ]:
# Download files (Google Colab)
# from google.colab import files
# !zip -r output.zip output/
# files.download('output.zip')

# Hoặc copy vào Google Drive
# !cp -r output/ /content/drive/MyDrive/BigData/output/

## 9. Tổng kết

### Kết quả đạt được:

1. **Thu thập dữ liệu**: Tải 9 bảng từ Kaggle (Brazilian E-Commerce)
2. **Làm sạch dữ liệu**: Xử lý missing values, duplicates, outliers
3. **Join & Tính toán**: Ghép 9 bảng thành 1 bảng fact, tính doanh thu theo nhiều chiều
4. **Machine Learning**: K-Means clustering phân cụm sản phẩm thành 4 nhóm
   - Premium: Sản phẩm doanh thu cao, cần ưu tiên tồn kho
   - Standard: Bán chạy, cần duy trì stock ổn định
   - Economy: Trung bình, quản lý bình thường
   - Low-value: Ít bán, cần xem xét giảm tồn kho
5. **Trực quan hóa**: 8+ biểu đồ phân tích
6. **Tối ưu Spark**: Cache, Repartition, Broadcast Join
7. **Export Power BI**: 8 files CSV sẵn sàng import

### Đề xuất tối ưu kho hàng:
- **Nhóm A (Premium)**: Tăng tồn kho, bố trí gần cửa ra -> giảm thời gian lấy hàng
- **Nhóm B (Standard)**: Duy trì mức tồn kho ổn định, reorder point tự động
- **Nhóm C (Economy/Low-value)**: Giảm tồn kho, đặt hàng theo lô nhỏ
- **Vận chuyển**: Tối ưu tuyến giao hàng cho các bang có tỉ lệ trễ cao

In [ ]:
# Dọn dẹp
spark.stop()
print("✅ SparkSession stopped. Done!")